# Look, It's a Fake! — Part II

**Competition**: UB Master FDS — Kaggle 2025/26 (Part II)
**Team**: Frenchies — Clara Bouvier, Bastien Laplace
**Result**: 2nd / 17

## Problem

Same binary classification task as Part I (LLM-generated vs. real content), two datasets:
- **Dataset A** (`derma`): tabular health/dermatology data — same features as Part I.
- **Dataset B** (`text`): news headlines — binary label (`fake` / `real`).

## Key Difference from Part I

Part II relaxes the model constraint: non-linear classifiers are allowed. The strategy for Dataset A becomes a **soft-voting ensemble** — CatBoost (gradient-boosted trees) blended with the Part I LogisticRegression, with weights optimized on the validation set. Dataset B upgrades from a single LogisticRegression to a **soft-voting ensemble** (LogisticRegression + RandomForest + LightGBM), paired with **rule-based data augmentation** to address class imbalance.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Dependencies

`CatBoostClassifier` is the key addition over Part I — it handles missing values natively and performs well on small tabular data without extensive preprocessing. `balanced_accuracy_score` is used as the blending optimization metric because it is robust to class imbalance, unlike raw accuracy.

In [ ]:
import numpy as np
import pandas as pd
import random
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, KBinsDiscretizer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin
from catboost import CatBoostClassifier
import lightgbm as lgb

## Data Loading

Four files: train/test splits for each dataset. Loaded from Google Drive (Colab environment).

In [ ]:
data_test_a_derma  = pd.read_csv(r'/content/drive/MyDrive/Kaggle_competition_its_a_fake/look-it-is-a-fake-2526-part-ii/test_A_derma.csv')
data_test_b_text   = pd.read_csv(r'/content/drive/MyDrive/Kaggle_competition_its_a_fake/look-it-is-a-fake-2526-part-ii/test_B_text.csv')
data_train_a_derma = pd.read_csv(r'/content/drive/MyDrive/Kaggle_competition_its_a_fake/look-it-is-a-fake-2526-part-ii/train_A_derma.csv')
data_train_b_text  = pd.read_csv(r'/content/drive/MyDrive/Kaggle_competition_its_a_fake/look-it-is-a-fake-2526-part-ii/train_B_text.csv')

---
## Dataset A — Preprocessing

Two preprocessing pipelines run in parallel on the same raw data, one per model:

- **`preprocess_dataset_a_v3`** (LogisticRegression): the full Part I pipeline — logical imputation, interaction features, KBins discretization of `Genetic Propensity` and `Skin X test`, missing indicators, mean imputation. Documented in Part I.
- **`preprocess_for_trees`** (CatBoost): a lighter version that drops the leaky feature and fills structural NaNs, but skips discretization and scaling. Tree-based models don't need normalized inputs and can learn non-monotonic relationships directly — discretization would only lose information.

Both pipelines operate on the same train/validation split indices so validation scores are directly comparable.

In [ ]:
LEAKY_FEATURE          = 'Doughnuts consumption'
TARGET_COLUMN          = 'Fake/Real'
ID_COLUMN              = 'Id'
NAN_FILL_VALUE_LESSION = 0.0

SIZE_COLS       = ['Small size', 'Mid size', 'Large size']
AREA_COLS       = ['Mid', 'Small', 'Large']
DISCRETIZE_COLS = ['Genetic Propensity', 'Skin X test']
N_BINS          = 5

def map_target(y_series):
    return y_series.map({'fake': 0, 'real': 1}).astype(int)

def apply_discretization(df, cols, n_bins, mode='train', discretizer=None):
    if mode == 'train':
        discretizer = KBinsDiscretizer(n_bins=n_bins, encode='onehot-dense', strategy='quantile', random_state=42)
        X_binned = discretizer.fit_transform(df[cols])
    else:
        X_binned = discretizer.transform(df[cols])

    if hasattr(discretizer, 'n_bins_'):
        bin_names = [f'{col}_bin_{i}' for col, n in zip(cols, discretizer.n_bins_) for i in range(n)]
    else:
        bin_names = [f'binned_feature_{i}' for i in range(X_binned.shape[1])]

    df_disc = pd.DataFrame(X_binned, columns=bin_names, index=df.index)
    df = pd.concat([df.drop(columns=cols, errors='ignore'), df_disc], axis=1)
    return df, discretizer

def preprocess_dataset_a_v3(df_raw, mode='train', train_imputer_values=None, train_nan_cols=None,
                             discretizer_obj=None, discretize_imputer_values=None):
    df     = df_raw.copy()
    df_ids = df[ID_COLUMN] if ID_COLUMN in df.columns else None
    df     = df.drop(columns=[ID_COLUMN, LEAKY_FEATURE], errors='ignore')

    if 'Lession' in df.columns:
        df['Lession'] = df['Lession'].fillna(NAN_FILL_VALUE_LESSION)

    for group in [SIZE_COLS, AREA_COLS]:
        for index, row in df.iterrows():
            if row[group].max() == 1.0:
                df.loc[index, group] = row[group].fillna(0.0)

    df['Propensity_X_Test']     = df['Genetic Propensity'] * df['Skin X test']
    df['Size_Area_Ratio_Proxy'] = (df['Small size'] + df['Mid size']) / (df['Small'] + df['Mid'] + 1e-6)
    df['Large_Coexistence']     = df['Large size'] * df['Large']

    if mode == 'train':
        discretize_imputer_values = df[DISCRETIZE_COLS].mean()
    df[DISCRETIZE_COLS] = df[DISCRETIZE_COLS].fillna(discretize_imputer_values)

    if mode == 'train':
        df, discretizer_obj = apply_discretization(df, DISCRETIZE_COLS, N_BINS, mode='train')
    else:
        df, _ = apply_discretization(df, DISCRETIZE_COLS, N_BINS, mode='test', discretizer=discretizer_obj)

    current_nan_cols = df.columns[df.isna().any()].tolist() if mode == 'train' else train_nan_cols

    for col in current_nan_cols:
        if col in df.columns:
            df[f'{col}_is_missing'] = df[col].isna().astype(int)

    if mode == 'train':
        imputer = df.mean()
        return df.fillna(imputer), imputer, current_nan_cols, df_ids, discretizer_obj, discretize_imputer_values
    else:
        df_out = df.fillna(train_imputer_values)
        return df_out.reindex(columns=train_imputer_values.index.tolist(), fill_value=0.0), df_ids

def preprocess_for_trees(df_raw):
    df = df_raw.copy()
    df = df.drop(columns=[ID_COLUMN, LEAKY_FEATURE], errors='ignore')

    if 'Lession' in df.columns:
        df['Lession'] = df['Lession'].fillna(NAN_FILL_VALUE_LESSION)

    for group in [SIZE_COLS, AREA_COLS]:
        mask = df[group].max(axis=1) == 1.0
        df.loc[mask, group] = df.loc[mask, group].fillna(0.0)

    df['Propensity_X_Test']     = df['Genetic Propensity'] * df['Skin X test']
    df['Size_Area_Ratio_Proxy'] = (df['Small size'] + df['Mid size']) / (df['Small'] + df['Mid'] + 1e-6)
    df['Large_Coexistence']     = df['Large size'] * df['Large']

    return df

### Training — Dataset A Ensemble

Three steps:

1. **Train LogisticRegression** on the preprocessed + scaled split, identical to Part I.
2. **Train CatBoost** with Optuna-tuned hyperparameters on the tree-friendly preprocessed split. `auto_class_weights='Balanced'` handles class imbalance. `early_stopping_rounds=200` prevents overfitting on the eval set.
3. **Grid-search blending weights**: iterate over all (CatBoost weight, LogReg weight) pairs in 1% increments, score the soft-vote blend on the held-out validation set using balanced accuracy, and keep the best combination.

Balanced accuracy is used instead of raw accuracy during weight optimization because it penalizes bias toward the majority class — a blend that always predicts the dominant class would score well on raw accuracy but poorly here.

In [ ]:
yA_full = map_target(data_train_a_derma[TARGET_COLUMN])
XA_raw  = data_train_a_derma.drop(columns=[TARGET_COLUMN], errors='ignore')

train_idx, val_idx = train_test_split(XA_raw.index, test_size=0.2, stratify=yA_full, random_state=42)
df_train_raw = XA_raw.loc[train_idx]
df_val_raw   = XA_raw.loc[val_idx]
y_train      = yA_full.loc[train_idx]
y_val        = yA_full.loc[val_idx]

X_train_log, imputer_values, nan_cols, _, discretizer_obj, disc_imputer = preprocess_dataset_a_v3(df_train_raw, mode='train')
X_val_log, _ = preprocess_dataset_a_v3(
    df_val_raw, mode='test',
    train_imputer_values=imputer_values, train_nan_cols=nan_cols,
    discretizer_obj=discretizer_obj, discretize_imputer_values=disc_imputer
)
scaler         = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_log)
X_val_scaled   = scaler.transform(X_val_log)

model_log = LogisticRegression(penalty=None, solver='lbfgs', max_iter=2000, random_state=42)
model_log.fit(X_train_scaled, y_train)

X_train_cat = preprocess_for_trees(df_train_raw)
X_val_cat   = preprocess_for_trees(df_val_raw)

best_cat_params = {
    'learning_rate':       0.0438,
    'depth':               4,
    'l2_leaf_reg':         7.211,
    'bagging_temperature': 0.621,
    'random_strength':     0.050,
    'border_count':        190,
    'iterations':          2000,
    'loss_function':       'Logloss',
    'eval_metric':         'Accuracy',
    'auto_class_weights':  'Balanced',
    'early_stopping_rounds': 200,
    'verbose':             0,
    'random_state':        42,
}

model_cat = CatBoostClassifier(**best_cat_params)
model_cat.fit(X_train_cat, y_train, eval_set=(X_val_cat, y_val), use_best_model=True)

probs_log_val = model_log.predict_proba(X_val_scaled)[:, 1]
probs_cat_val = model_cat.predict_proba(X_val_cat)[:, 1]

best_w_cat, best_score = 0.0, 0.0
for w_cat in np.linspace(0, 1, 101):
    blend = probs_cat_val * w_cat + probs_log_val * (1.0 - w_cat)
    score = balanced_accuracy_score(y_val, (blend > 0.5).astype(int))
    if score > best_score:
        best_score, best_w_cat = score, w_cat

print(f"Best balanced accuracy: {best_score:.4f}")
print(f"Optimal weights — CatBoost: {best_w_cat:.2f} | LogReg: {1.0 - best_w_cat:.2f}")

X_test_log, test_ids_a = preprocess_dataset_a_v3(
    data_test_a_derma, mode='test',
    train_imputer_values=imputer_values, train_nan_cols=nan_cols,
    discretizer_obj=discretizer_obj, discretize_imputer_values=disc_imputer
)
X_test_scaled  = scaler.transform(X_test_log)
X_test_cat     = preprocess_for_trees(data_test_a_derma)

probs_test_log = model_log.predict_proba(X_test_scaled)[:, 1]
probs_test_cat = model_cat.predict_proba(X_test_cat)[:, 1]
final_probs_a  = probs_test_cat * best_w_cat + probs_test_log * (1.0 - best_w_cat)
final_preds_a  = (final_probs_a > 0.5).astype(int)

label_map_inv = {0: 'fake', 1: 'real'}
submission_df_A = pd.DataFrame({
    'Id':       test_ids_a.values,
    'Fake/Real': pd.Series(final_preds_a).map(label_map_inv).values
})
print(submission_df_A.head())
print(f"Dataset A submission shape: {submission_df_A.shape}")

---
## Dataset B — Data Augmentation

The training set for Dataset B is class-imbalanced. A rule-based generator creates synthetic fake headlines by:
1. Applying lexical substitution rules to existing fake titles (e.g. "Study" → "Secret Report", "Says" → "Claims") to inject sensationalist vocabulary.
2. Appending a sensational suffix for titles that don't match any substitution rule, and randomly for 30% of those that do.

The target is **1400 fake titles** total. Augmented titles are not realistic headlines, but they reinforce the exact stylistic patterns the classifier is learning to detect.

In [ ]:
df_b_train  = data_train_b_text.copy()
fake_titles = df_b_train[df_b_train['Fake/Real'].str.strip() == 'fake']['Title'].tolist()

N_TOTAL_FAKE  = 1400
N_TO_GENERATE = N_TOTAL_FAKE - len(fake_titles)

replacement_rules = {
    r'\bNew\b':          'Shocking',
    r'\bStudy\b':        'Secret Report',
    r'\bMajor\b':        'Massive',
    r'\bPredicts\b':     'Unveils',
    r'\bSays\b':         'Claims',
    r'\bFinds\b':        'Exposes',
    r'\bLaunches\b':     'Unveils',
    r'\bPlan\b':         'Secret Agenda',
    r'\bControversy\b':  'Scandal',
    r'\bUpdate\b':       'Urgent Alert',
    r'\bGuide\b':        'Hidden Truths',
    r'\bDevelops\b':     'Confirms',
    r'\bReport\b':       'Breaking Revelation',
    r'\bReveals\b':      'Exposes',
}

SENSATIONAL_PHRASES = [
    ' That Will Blow Your Mind!',
    " You Won't Believe!",
    ' SHOCKING Details Inside!',
    ' And The World Is Silent!',
    ' EXCLUSIVE Footage!',
    ' Revealed!',
]

def augment_title(title, rules):
    text = title.strip()
    rule_items = list(rules.items())
    random.shuffle(rule_items)
    applied = False
    for pattern, replacement in rule_items:
        if re.search(pattern, text, re.IGNORECASE):
            def replace_case(match):
                w = match.group(0)
                if w.isupper():  return replacement.upper()
                if w.istitle():  return replacement.title()
                if w.islower():  return replacement.lower()
                return replacement
            text    = re.sub(pattern, replace_case, text, count=1, flags=re.IGNORECASE)
            applied = True
            break
    if not applied or random.random() < 0.3:
        suffix = random.choice(SENSATIONAL_PHRASES)
        text   = text + suffix if not text.endswith(('!', '.', '?')) else text + ' Must See!'
    return text

def merge_datasets(original, new):
    return pd.concat([original, new], ignore_index=True)

random.seed(42)
sampled  = random.choices(fake_titles, k=N_TO_GENERATE)
new_data = pd.DataFrame({
    'Id':       range(df_b_train['Id'].max() + 1, df_b_train['Id'].max() + 1 + N_TO_GENERATE),
    'Title':    [augment_title(t, replacement_rules) for t in sampled],
    'Fake/Real': 'fake',
})

print(f"Original: {len(df_b_train)} | Generated: {len(new_data)}")
print(f"Total after merge: {len(df_b_train) + len(new_data)}")

### NLP Pipeline — Dataset B

The same triple vectorizer as Part I (word TF-IDF, char CountVectorizer, char_wb TF-IDF), but two things change:

**`StyleFeatureExtractor`** replaces `GranularFeatureTransformer`. It captures five raw counts — capitals, punctuation (`!?.`), title length, word count, digit count — rather than the targeted keyword flags used in Part I. More generalizable, less brittle to vocabulary shifts.

**Soft voting ensemble** (LogReg + RandomForest + LightGBM) replaces the single LogReg. Three structurally diverse classifiers operating on the same feature matrix: a linear model (LogReg), a bagged ensemble of trees (RandomForest), and a gradient-boosted ensemble (LightGBM). Averaging their probability outputs reduces variance and catches patterns that any single model misses.

The model is validated on a stratified 80/20 split, then retrained on the full augmented dataset before generating test predictions.

In [ ]:
class StyleFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.astype(str)
        capital_count     = X.apply(lambda x: sum(1 for c in x if c.isupper()))
        punctuation_count = X.apply(lambda x: len(re.findall(r'[!?.]', x)))
        title_length      = X.apply(len)
        word_count        = X.apply(lambda x: len(x.split()))
        digit_count       = X.apply(lambda x: sum(c.isdigit() for c in x))
        return pd.DataFrame({
            'capital_count':     capital_count,
            'punctuation_count': punctuation_count,
            'title_length':      title_length,
            'word_count':        word_count,
            'digit_count':       digit_count,
        })


preprocessor = ColumnTransformer(
    transformers=[
        ("word",          TfidfVectorizer(max_features=20000, ngram_range=(1, 4), stop_words="english", min_df=2, max_df=0.95, sublinear_tf=True), 'Title'),
        ("char",          CountVectorizer(analyzer="char", ngram_range=(2, 4), min_df=2, max_features=20000, lowercase=False), 'Title'),
        ("subword_tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=20000, sublinear_tf=True), 'Title'),
        ("numeric_flags", StyleFeatureExtractor(), 'Title'),
    ],
    remainder="drop"
)

clf1 = LogisticRegression(max_iter=1000, random_state=42)
clf2 = RandomForestClassifier(random_state=42, class_weight='balanced')
clf3 = lgb.LGBMClassifier(random_state=42, class_weight='balanced')

voting_clf = VotingClassifier(
    estimators=[('lr', clf1), ('rf', clf2), ('lgb', clf3)],
    voting='soft'
)

final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   voting_clf)
])

merged_data = merge_datasets(data_train_b_text, new_data)
merged_data.columns          = merged_data.columns.str.strip()
data_test_b_text.columns     = data_test_b_text.columns.str.strip()
merged_data                  = merged_data.dropna(subset=['Fake/Real'])
data_test_b_text             = data_test_b_text.dropna(subset=['Title'])
merged_data['Fake/Real']     = merged_data['Fake/Real'].map({'fake': 0, 'real': 1})

X_train_split, X_val, y_train_split, y_val = train_test_split(
    merged_data[['Title']],
    merged_data['Fake/Real'],
    test_size=0.2,
    random_state=42,
    stratify=merged_data['Fake/Real']
)

final_pipeline.fit(X_train_split, y_train_split)
y_val_pred = final_pipeline.predict(X_val)

print("--- Ensemble Validation Results ---")
print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(classification_report(y_val, y_val_pred, target_names=['fake', 'real']))

final_pipeline.fit(merged_data[['Title']], merged_data['Fake/Real'])

test_predictions = final_pipeline.predict(data_test_b_text[['Title']])
submission_df_B  = pd.DataFrame({
    'Id':       data_test_b_text['Id'],
    'Fake/Real': pd.Series(test_predictions).map({0: 'fake', 1: 'real'})
})
print(submission_df_B.head())
print(f"Dataset B submission shape: {submission_df_B.shape}")

---
## Submission

Dataset A and Dataset B predictions are merged into a single file. Dataset B IDs are offset by the number of Dataset A rows to avoid collisions.

In [ ]:
def merge_submission_dfs(df_a, df_b, sample_submission_path=None, output_path="submission_final.csv"):
    for df_name, df in zip(["A", "B"], [df_a, df_b]):
        if "Fake/Real" not in df.columns:
            raise ValueError(f"DataFrame {df_name} is missing column 'Fake/Real'.")
        if "Id" not in df.columns:
            raise ValueError(f"DataFrame {df_name} is missing column 'Id'.")

    offset = df_a["Id"].max() + 1
    df_b   = df_b.copy()
    df_b["Id"] = df_b["Id"] + offset

    submission_final = pd.concat([df_a, df_b], ignore_index=True)
    submission_final = submission_final.sort_values(by="Id").reset_index(drop=True)

    if sample_submission_path is not None:
        try:
            sample = pd.read_csv(sample_submission_path)
            if len(submission_final) != len(sample):
                print(f"Warning: {len(submission_final)} rows generated, {len(sample)} expected.")
            else:
                print("Row count matches sample_submission.csv")
        except Exception as e:
            print(f"Could not read {sample_submission_path}: {e}")

    submission_final.to_csv(output_path, index=False)
    print(f"Merged: {len(df_a)} (A) + {len(df_b)} (B) = {len(submission_final)} rows")
    print(f"Saved: {output_path}")
    return submission_final

### Generate Final Submission File

In [ ]:
final_submission = merge_submission_dfs(
    submission_df_A,
    submission_df_B,
)

---
## Results & Takeaways

**Final leaderboard**: 2nd / 17

### Validation scores

| Dataset | Model | Score |
|---|---|---|
| A — Dermatology | LogReg alone | 0.8083 (balanced acc.) |
| A — Dermatology | CatBoost alone | 0.8372 (balanced acc.) |
| A — Dermatology | Ensemble (CatBoost 0.98 + LogReg 0.02) | **0.8547** (balanced acc.) |
| B — News Headlines | LogReg + RF + LightGBM ensemble + augmentation | **0.9403** (accuracy) |

**Dataset B — per-class breakdown (validation set, n=1189):**

| Class | Precision | Recall | F1 |
|---|---|---|---|
| fake | 0.88 | 0.93 | 0.90 |
| real | 0.97 | 0.95 | 0.96 |
| **macro avg** | **0.92** | **0.94** | **0.93** |

### Changes vs. Part I

| Component | Part I | Part II |
|---|---|---|
| Dataset A model | LogReg only | CatBoost + LogReg soft-vote ensemble |
| Dataset A weight optimization | N/A | Grid search on balanced accuracy |
| Dataset B feature extractor | `GranularFeatureTransformer` (keyword flags) | `StyleFeatureExtractor` (raw counts) |
| Dataset B classifier | Single LogReg | Soft-vote ensemble: LogReg + RF + LightGBM |
| Dataset B training data | Original | Rule-based augmented (→ 1400 fake titles) |

### Decisions that mattered

- **CatBoost as the dominant model for Dataset A**: the optimal blend weight was 0.98 CatBoost / 0.02 LogReg — essentially pure CatBoost. Trees naturally handle missing values and non-linear interactions in the dermatology data without the discretization workarounds required for LogReg. The marginal LogReg weight suggests it adds a small stabilizing signal but CatBoost carries the prediction.
- **Optuna for CatBoost tuning**: CatBoost's hyperparameter space is too large for manual grid search. Optuna's TPE sampler finds good hyperparameter regions efficiently in far fewer evaluations.
- **Balanced accuracy for Dataset A weight selection**: on an imbalanced validation set, optimizing raw accuracy gravitates toward predicting the majority class. Balanced accuracy forces equal weight on both classes.
- **Soft voting ensemble for Dataset B**: combining three structurally diverse classifiers (linear, bagged trees, gradient boosting) on the same feature matrix reduces variance. Each model captures different aspects of the fake/real signal.
- **`StyleFeatureExtractor` over keyword flags**: raw stylometric counts (length, capitals, punctuation) generalize better across the augmented distribution than targeted keyword lists, which risk overfitting to the specific vocabulary introduced by the augmentation rules.
- **Rule-based augmentation**: even low-quality synthetic headlines help when the learned signal is stylistic. The substitution rules reinforce the vocabulary patterns that distinguish LLM-generated from real headlines.